## Interior point method

In [6]:
import numpy as np
import pandas as pd
import copy as copy
import scipy.io
import time
import os
from scipy.linalg import solve, LinAlgWarning
import warnings
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import openpyxl
import xlsxwriter

from inpoint_methods import paso_intpoint, loadProblem, intpoint, intpointR, highlight_greaterthan #,intpointR_mask

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [15]:
def solve_catch(K,ld):
    with warnings.catch_warnings(record=True) as warneo:
        warnings.simplefilter("always", LinAlgWarning)
        # Solve the linear system
        w_vector = scipy.linalg.solve(K, ld)
        
        # Check if any warning was triggered
        if any(issubclass(warn.category, LinAlgWarning) for warn in warneo):
            # Print the warning message for all warnings captured
            for warn in warneo:
                print(f"Warning: {warn.message}")
    return w_vector

def highlight_df(mu, z, Q, k, tau, highlighted_df, mudf, epsilon=1e-6, comp_tol=1e-5):
    # Previous μ from mudf
    prev_mu = mudf.loc[k-1].values
    
    # Only consider indices where μ*z <= comp_tol
    mask = mu * z <= comp_tol
    
    # Highlight if μ did not increase significantly and μ*z is small enough
    highlighted_rows = [i for i in range(len(mu)) if mask[i] and mu[i] <= prev_mu[i] + epsilon]
    
    # Build row of 1s and 0s
    highlighted_row = [1 if i in highlighted_rows else 0 for i in range(Q.shape[0])]
    
    # Store in DataFrame
    highlighted_df.loc[k] = highlighted_row
    return highlighted_df

def display_highlights(highlighted_df,regression_df):
    # Take last 3 iterations
    last_3_iterations = highlighted_df.tail(3)
    
    # Columns consistently highlighted (1 in all last 3 iterations)
    highlighted_columns = last_3_iterations.columns[(last_3_iterations == 1).all(axis=0)].tolist()
    highlighted_columns = [int(x) for x in highlighted_columns]  # ensure ints
    print("Consistently highlighted in last 3 iterations:", highlighted_columns)
    print("How many?=", len(highlighted_columns))
    
    # Columns that were highlighted in previous iteration but regressed this iteration (1 → 0)
    regressed_columns = []
    if len(highlighted_df) >= 2:
        prev_iter = highlighted_df.iloc[-2]
        last_iter = highlighted_df.iloc[-1]
        regressed_columns = prev_iter.index[(prev_iter == 1) & (last_iter == 0)].tolist()
        regressed_columns = [int(x) for x in regressed_columns]
        if regressed_columns:
            print(f"WARNING: Iter {highlighted_df.index[-1]} regressed columns (1 -> 0):", regressed_columns)

            highlighted_df_regressed = highlighted_df.tail(10)[regressed_columns]
            display(highlighted_df_regressed)
        iter_idx = highlighted_df.index[-1]
        if regression_df is not None:
                row = pd.Series(0, index=highlighted_df.columns)
                row[regressed_columns] = 1
                regression_df.loc[iter_idx] = row
    
    return highlighted_columns, regressed_columns


def progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                 main_norm, pr_before, inq_before, cmp_before,
                                 tau_before, cond_before, obj_before):
    """
    Return a DataFrame with the progress summary and an 'Improved?' column.
    Marks True if the metric improved after the step.
    """

    # Compute "after" values
    pr_after   = np.linalg.norm(v2, np.inf)
    inq_after  = np.linalg.norm(v3, np.inf)
    cmp_after  = np.max(v4)
    tau_after  = tau
    cond_after = np.linalg.cond(G, 1)
    obj_after  = 0.5 * x @ Q @ x + c @ x
    norm_after = np.linalg.norm(ld1, np.inf)

    # Define metrics, before and after
    metrics = [
        ("overall ||ld||∞", main_norm, norm_after),
        ("primal ||·||∞", pr_before, pr_after),
        ("ineq ||·||∞", inq_before, inq_after),
        ("max(mu*z)", cmp_before, cmp_after),
        ("tau", tau_before, tau_after),
        ("cond(G)", cond_before, cond_after),
        ("objective", obj_before, obj_after)
    ]

    # Build table with improvement
    summary_data = {
        "Metric": [],
        "Before": [],
        "After": [],
        "Improved?": []
    }

    for name, before, after in metrics:
        summary_data["Metric"].append(name)
        summary_data["Before"].append(before)
        summary_data["After"].append(after)
        
        # Determine improvement
        # Smaller is better for residuals, complementarity, tau, cond(G)
        # For objective, smaller is better if minimizing
        if name in ["overall ||ld||∞", "primal ||·||∞", "ineq ||·||∞", "max(mu*z)", "tau", "cond(G)", "objective"]:
            improved = after < before
        else:
            improved = np.nan  # unknown metric

        summary_data["Improved?"].append(improved)

    summary_df = pd.DataFrame(summary_data)

    return summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after

def build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns):
    # Build block rows
    m = A.shape[0]
    p = U.shape[0]
    n = Q.shape[0]

    r1 = np.hstack((Q, AT, -FT @ U))
    r2 = np.hstack((A, np.zeros((m, m + p))))
    r3 = np.hstack((-U @ F, np.zeros((p, m)), -Z @ U))

    M1 = np.vstack((r1, r2, r3))

    # Step 1: Filter mu
    mu_filtered = np.array([mu[i] for i in range(len(mu)) if i not in highlighted_columns])

    # Step 2: Mapping matrix (positions old -> new)
    p_filtered = len(mu_filtered)
    mapping_matrix = np.zeros((p, p_filtered), dtype=int)
    for new_pos, old_pos in enumerate([i for i in range(p) if i not in highlighted_columns]):
        mapping_matrix[old_pos, new_pos] = 1

    # Step 3: Build U1
    U1 = np.diag(mu_filtered)

    # Step 4: Remove rows and columns
    rows_to_remove = [i + (n + m) for i in highlighted_columns]
    num_removed = len(rows_to_remove)
    M1 = np.delete(M1, rows_to_remove, axis=0)
    M1 = np.delete(M1, rows_to_remove, axis=1)

    print(f"Deleted {num_removed} rows/columns. New M1 shape: {M1.shape}")

    if M1.shape[0] != M1.shape[1]:
        raise ValueError("M1 is not square! Check highlighted indices.")

    # Step 5: Build reduced RHS vector
    ld1 = np.concatenate((
        Q @ x + AT @ lamda - FT @ mu + c,
        A @ x - b,
        U @ (d - F @ x) + tau
    ))
    ld1 = np.delete(ld1, rows_to_remove, axis=0)

    return M1, U1, ld1

def load_lp_problem(mat_file: str):
    """
    Loads a linear/quadratic problem from a .mat file and initializes
    the problem matrices for the interior-point algorithm.
    
    Returns:
        Q : ndarray    Quadratic matrix (identity by default)
        c : ndarray    Linear term vector
        A : ndarray    Equality constraint matrix
        b : ndarray    Equality constraint RHS
        F : ndarray    Inequality constraint matrix (identity by default)
        d : ndarray    Inequality constraint RHS (zeros)
        H : dict       Raw data loaded from the .mat file
    """
    print(f"Loading problem from: {mat_file}")
    H = loadProblem(f"mat_files/{mat_file}")

    # Quadratic term: identity (can be changed if needed)
    Q = np.eye(H['c'].shape[0])

    # Linear term
    c = H['c'].ravel()  # flatten in case it's a column vector

    # Equality constraints
    A = H['AE']
    b = H['bE'].ravel()  # flatten in case it's a column vector

    # Compute percentage of zeros in b
    num_zeros_b = np.sum(b == 0)
    pct_zeros_b = 100 * num_zeros_b / len(b)
    
    # Inequality constraints (x >= 0 by default)
    F = np.eye(H['c'].shape[0])
    d = np.zeros(H['c'].shape[0])

    print(f"Problem loaded. n={Q.shape[0]}, m={A.shape[0]}, p={F.shape[0]}")
    print(f"Equality RHS b: {b}")
    print(f"Number of zeros in b: {num_zeros_b} ({pct_zeros_b:.2f}%)")

    return Q, c, A, b, F, d, H


Load problem

In [8]:
#mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

print(mat_files)

H = loadProblem(f"mat_files/{mat_files}")

# Define the quadratic matrix Q
Q = np.eye(H['c'].shape[0])

# Define the linear term vector c
c = H['c']

# Define the equality constraint matrix A and vector b
A = H['AE']
b = H['bE']
#b = np.zeros( A.shape[0] )

# Define an inequality constraint: x1 >= -10
#F = np.zeros([H['c'].shape[0],H['c'].shape[0]])
F = np.eye(H['c'].shape[0])
d = np.zeros([H['c'].shape[0],1])
d = d.ravel()  # Flattens the vector to a 1D vector
print(b)

lp_blend.mat
 Norma infinita de b:  26.32
[ 0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.    0.
  0.    0.    0.    0.   23.26  5.25 26.32 21.05 13.45  2.58 10.   10.
  0.    0.  ]


In [ ]:
# Sistema Completo
#x, lamda, mu, z, k = intpoint(Q, A, F, c, b, d)
# Sistema Reducido
#x, lamda, mu, z, k = intpointR(Q, A, F, c, b, d)

# Método de puntos interiores con heurística (¿?)

In [ ]:
# Choose a file
mat_files = 'lp_kb2.mat'     # b = 0
#mat_files = 'lp_afiro.mat'   # bmax = 500, and no mu tends to zero  and no condition problem
#mat_files = 'lp_blend.mat'    # bmax = 26.32, and no mu tends to zero.
#mat_files = 'lp_fit1d.mat'   # b = 0
#mat_files = 'lp_fit1p.mat'   # bmax = 216, and no mu tends to zero, condition 10^12
#mat_files = 'lp_grow15.mat'  # b = 0, condition 10^6
#mat_files = 'lp_grow22.mat'  # b = 0, condition 1.4e7
#mat_files = 'lp_grow7.mat'   # b = 0, condition 1.2e7

# Load problem
Q, c, A, b, F, d, H = load_lp_problem(mat_files)

k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

problem_info = {
    "problem_name": mat_files,
    "norm_inf_b": np.linalg.norm(b, np.inf),
    "dim_b": len(b),
    "num_zeros_b": np.sum(b==0),
    "dim_x": n,
    "dim_eq": m,
    "dim_ineq": p
}

tol = 1e-6
kmax = 100

#print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
#z = np.ones(p)
z = F @ x - d + (0.5)*np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=range(p))
zdf = pd.DataFrame(columns=range(p))
taudf = pd.DataFrame(columns=range(p))
ratiodf = pd.DataFrame(columns=range(p))
highlighted_df = pd.DataFrame(columns=range(p))

#Dataframes which record mu, z and tau over each iteration
mudf.loc[k] = mu
zdf.loc[k] = z
taudf.loc[k] = np.full(p, tau)
ratiodf.loc[k] = mu / tau

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld1 = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld1,np.inf) # this is the CNPO without the perturbation
w = np.zeros((p, 1))

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns
regression_df = pd.DataFrame(columns=highlighted_df.columns) #store any regressed indexes for mu

red_mu = []

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    M = np.vstack((row1, row2, row3, row4))
    
    D = np.diag(mu / z)
    G = Q+FT@D@F
    for i in range(p):
        w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
    w = w.ravel()
        
    dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    m = A.shape[0]
    K = np.block([
        [G, AT],
        [A, np.zeros((m, m))]
    ])
    
    # Calculate the reciprocal condition number of G
    condK = np.linalg.cond(G,1)
    
    # Define ld
    ld = -np.concatenate([dg, A @ x - b])
    #norma_cnpo = np.linalg.norm(ld, np.inf)
    
    # Solve the linear system and catch errors

    w_vector = solve_catch(K,ld) # Reduced system
    
    # Update the sections of the w_vector
    wx     = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu    = - D @ (F @ wx + w)
    wz     = - ( (1 / mu) * (z * wmu - tau) + z )
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    alpha    = 0.995 * min(alpha_mu, alpha_z)
    #alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # remember something
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1

    mudf.loc[k] = mu    # dataframe for graphing the central path
    zdf.loc[k] = z  
    taudf.loc[k] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[k] = mu / tau

    highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)

    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)

    main_norm = norma_cnpo # CHECK IF THIS IS CORRECT HERE
    pr_before   = np.linalg.norm(v2, np.inf) # before name is misleading
    inq_before  = np.linalg.norm(v3, np.inf) # before name is misleading
    cmp_before  = np.max(v4) # before name is misleading
    tau_before  = tau # before name is misleading
    cond_before = condK # before name is misleading
    obj_before  = 0.5 * x @ Q @ x + c @ x # before name is misleading
    
    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])

    if all(mask): #Return True only if every entry in mask is True
        # Once the mu and z have gotten sufficiently small,
        # we record where we were before we start with the heuristic/experiment:

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu, z, Q, k, tau, highlighted_df, mudf)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix, as we already exited the while loop

print("final k= ",k)
k_red = 0
tol_red = 1e-10
max_red = 10
#print("CHECK BEFORE REDUCTION")
highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)
regressed_df = pd.DataFrame(columns=['Iteration', 'Regressed_columns'])

while k_red < max_red:
#while np.linalg.norm(ld1, np.inf) > tol_red and k_red < max_red:
    Z = np.diag(z)
    U = np.diag(mu)

    M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

    print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

    # SOLVE THE NEW SYSTEM
    w_vector1 = scipy.linalg.solve(M1, ld1)

    wx1     = w_vector1[:n] # should be the same ?
    wlamda1 = w_vector1[n:n + m] # should be the same ?
    Dwmu1    = w_vector1[n + m:]
    
    wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

    # Here's the vector transformed back into the original size problem
    full_wmu = np.zeros_like(mu)
    active_indices = [i for i in range(p) if i not in highlighted_columns]
    for i, idx in enumerate(active_indices):
        full_wmu[idx] = wmu1[i]

    # ────────── complete here ──────────
    # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
    # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
    full_wz = F @ wx1 - (F @ x - d - z)

    # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
    alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
    alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
    alpha    = 0.995 * min(alpha_mu, alpha_z)

    # Compute perc_mu, perc_z for the reduced step
    perc_mu = full_wmu / mu
    perc_z  = full_wz / z

    # 2. Apply the step with the same step‑size alpha
    x      += alpha * wx1
    lamda  += alpha * wlamda1
    mu     += alpha * full_wmu
    z      += alpha * full_wz

    # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
    tau = sigma * np.dot(mu, z) / p

    #print("max change in x:", np.max(np.abs(alpha*wx1)))
    #print("max change in mu:", np.max(np.abs(alpha*full_wmu)))
    #print("max change in z:", np.max(np.abs(alpha*full_wz)))

    # 3. Recompute residuals for diagnostics
    v1 = Q @ x + AT @ lamda - FT @ mu + c      # dual residual
    v2 = A @ x - b                             # primal residual
    v3 = -F @ x + d + z                        # inequality residual
    v4 = mu * z                                # complementarity product
    ld1 = np.concatenate((v1, v2, v3, v4))

    norm_after = np.linalg.norm(ld1, np.inf)

    k_red += 1
    total_iter= k + k_red
    #print("total_iter:", total_iter)

    # Update highlighted_df after this reduced iteration

    mudf.loc[total_iter] = mu    # dataframe for graphing the central path
    zdf.loc[total_iter] = z  
    taudf.loc[total_iter] = np.full(p, tau)     # dataframe for graphing the central path
    ratiodf.loc[total_iter] = mu / tau
    
    highlighted_df = highlight_df(mu, z, Q, total_iter, tau, highlighted_df, mudf)

    # Optionally, recompute the highlighted columns for the next reduced iteration
    highlighted_columns, regressed_columns = display_highlights(highlighted_df,regression_df)

    # AFTER computing highlighted_columns, regressed_columns in the reduced loop:

    if len(regressed_columns) > 0:
        regressed_df = pd.concat([regressed_df,
                                pd.DataFrame({'Iteration': [k_red],
                                                'Regressed_columns': [regressed_columns]})],
                                ignore_index=True)
        print(f"Iteration of reduced loop {k_red}: Regressed columns:", regressed_columns)
    else:
        print(f"Iteration of reduced loop  {k_red}: No regressed columns")

summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after = \
        progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                main_norm, pr_before, inq_before, cmp_before,
                                tau_before, cond_before, obj_before)

display(summary_df)

# Save to excel file
# After finishing the loops
# After your existing computation and summary_df creation:

# Add problem info
summary_df['Problem'] = mat_files
summary_df['dim_x'] = n
summary_df['dim_A'] = m
summary_df['dim_F'] = p
summary_df['||b||_inf'] = np.linalg.norm(b, np.inf)
summary_df['num_zeros_in_b'] = np.sum(b == 0)

# Reorder columns if you want
cols_order = ['Problem', 'dim_x', 'dim_A', 'dim_F', '||b||_inf', 'num_zeros_in_b', 'Metric', 'Before', 'After']
summary_df = summary_df[cols_order]

# After finishing all computations:

# After all your iterations and building summary_df and regressed_df

with pd.ExcelWriter(f"progress_summary_{mat_files}.xlsx", engine='xlsxwriter') as writer:
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    if 'regressed_df' in locals() and not regressed_df.empty:
        regressed_df.to_excel(writer, sheet_name='Regressions', index=False)

print(f"Progress summary with regressions saved for {mat_files}")

print("Condition number (1-norm):", cond_after)
print("Objective after:", obj_after)

Loading problem from: lp_kb2.mat
 Norma infinita de b:  0.0
Problem loaded. n=68, m=43, p=68
Equality RHS b: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Number of zeros in b: 43 (100.00%)

iter= 1 	 ||cnpo||= 490.33642370558937
Condition number of G: 1.0
rcond(G) 1.0
tau = 1.2168445245304944

iter= 2 	 ||cnpo||= 76.98594752428107
Condition number of G: 88.24009902077839
rcond(G) 0.011332716203826159
tau = 0.5203292093316584

iter= 3 	 ||cnpo||= 29.44463736834997
Condition number of G: 87.18100139634159
rcond(G) 0.011470389006588807
tau = 0.3099010872970435

iter= 4 	 ||cnpo||= 19.80392696585398
Condition number of G: 3213.4944891031523
rcond(G) 0.00031118771275039217
tau = 0.2504851514766932

iter= 5 	 ||cnpo||= 8.854125671116936
Condition number of G: 17381.94745417208
rcond(G) 5.753095288295652e-05
tau = 0.16022449598782154

iter= 6 	 ||cnpo||= 3.492077279913038
Condition number of G: 42431.199142

,22,26,45,60
16,0,0,0,0
17,0,0,0,0
18,0,0,0,0
19,0,0,0,0
20,0,0,0,0
21,0,0,0,0
22,1,1,1,1
23,1,1,1,1
24,1,1,1,1
25,0,0,0,0


Deleted 62 rows/columns. New M1 shape: (117, 117)
cond(M1, 1-norm): 126383068.0496026
Consistently highlighted in last 3 iterations: [0, 2, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67]
How many?= 62
Iteration of reduced loop  1: No regressed columns
Deleted 62 rows/columns. New M1 shape: (117, 117)
cond(M1, 1-norm): 126243187.07197997
Consistently highlighted in last 3 iterations: [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 25, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 61, 62, 63, 64, 65, 66, 67]
How many?= 63
Iteration of reduced loop  2: No regressed columns
Deleted 63 rows/columns. New M1 shape: (116, 116)
cond(M1, 1-norm): 126244345.19566022
Consistently highlighted in las

,Metric,Before,After,Improved?
0,overall ||ld||∞,8.803881e-07,1.041735e-06,False
1,primal ||·||∞,3.469447e-17,4.435688e-15,False
2,ineq ||·||∞,1.387779e-17,1.776357e-15,False
3,max(mu*z),8.803881e-07,1.041735e-06,False
4,tau,3.806842e-07,3.769465e-07,True
5,cond(G),1.024103e+08,1.024103e+08,False
6,objective,-2.081026e-02,-2.081077e-02,True


Progress summary with regressions saved for lp_kb2.mat
Condition number (1-norm): 102410293.2530367
Objective after: -0.020810768945027095
